# Session 2 · Prepare data (Colab)

Turns FineWeb-Edu sample-10BT (14 files, 28.5 GB) into about 10B tokens of shards on your Google Drive, plus a report.

- **Runtime:** any; the work is CPU-heavy, so more CPU cores means faster. An L4 or A100 runtime has more cores than the basic CPU one.
- **Drive space:** about 20 GB of shards; cell 2 checks this.
- **Resumable:** if Colab disconnects, reopen this notebook and *Run all*. Finished files are skipped.
- Keep the tab open (or use Colab Pro+ background execution). It takes a few hours.
- **If Colab says "restart session" or reconnects:** just run cell 1 again, then continue. Every later cell uses full paths.

In [1]:
# 1. Keys, Drive, latest code, packages
import os, subprocess
from google.colab import userdata, drive
for k in ['HF_TOKEN','WANDB_API_KEY','GIT_NAME','GIT_EMAIL','GITHUB_TOKEN']:
    try: os.environ[k] = userdata.get(k)
    except Exception: pass
drive.mount('/content/drive')
os.environ['CHOTU_STORAGE'] = '/content/drive/MyDrive/chotu'
os.makedirs(os.environ['CHOTU_STORAGE'], exist_ok=True)
tok = os.environ.get('GITHUB_TOKEN')
url = f'https://{tok}@github.com/aditya950820/Chotu.git' if tok else 'https://github.com/aditya950820/Chotu.git'
if not os.path.exists('/content/Chotu'): subprocess.run(['git','clone',url,'/content/Chotu'],check=True)
%cd /content/Chotu
!git pull --rebase -q && git config user.name "$GIT_NAME" && git config user.email "$GIT_EMAIL"
# Keep Colab's own torch, numpy, pyarrow, huggingface_hub: replacing them forces a restart or breaks Colab's other packages
!cd /content/Chotu && grep -vE '^(torch|numpy|pyarrow|huggingface_hub)==' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt
import multiprocessing as mp
n = mp.cpu_count()
print(f'CPU cores: {n}  ->  full run roughly {28/max(n,1):.0f} h (about 0.1M tokens/s per core)')
if n < 8: print('Few cores: Runtime -> Change runtime type -> A100 (or v2-8 TPU) for more cores; the run is resumable either way.')

Mounted at /content/drive
/content/Chotu
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.
CPU cores: 2  ->  full run roughly 14 h (about 0.1M tokens/s per core)
Few cores: Runtime -> Change runtime type -> A100 (or v2-8 TPU) for more cores; the run is resumable either way.


In [2]:
# 2. Drive space check (need ~22 GB free for shards)
import os, shutil
free = shutil.disk_usage('/content/drive/MyDrive').free/1e9
print(f'Drive free: {free:.1f} GB'); assert free > 22, 'Not enough Drive space: upgrade Google One storage or free up space.'

Drive free: 71.4 GB


In [3]:
# 3. Quick test (~1 min): tokenizer checks + 2 small shards in chotu/test/
# Self-contained: works even after a session restart
!cd /content/Chotu && python scripts/prepare_data.py --storage /content/drive/MyDrive/chotu --max-docs 2000 --max-files 2

config.json: 100% 704/704 [00:00<00:00, 3.51MB/s]
tokenizer_config.json: 100% 3.66k/3.66k [00:00<00:00, 7.92MB/s]
vocab.json: 100% 801k/801k [00:00<00:00, 29.2MB/s]
merges.txt: 100% 466k/466k [00:00<00:00, 34.2MB/s]
special_tokens_map.json: 100% 831/831 [00:00<00:00, 5.26MB/s]
tokenizer.json: 100% 2.10M/2.10M [00:00<00:00, 144MB/s]
2 source files, storage=/content/drive/MyDrive/chotu/test, workers=2
[OK] round trip: 200/200 documents identical after encode-decode
[OK] digits split: 2048 -> ['2', '0', '4', '8']
[OK] <|im_start|> single token: ids=[1]
[OK] <|im_end|> single token: ids=[2]
[OK] <|endoftext|> single token: ids=[0]
[OK] end-of-text id is 0: 
[OK] vocab fits uint16: vocab=49,152
[1/2] sample/10BT/000_00000.parquet: already done, skipping
[2/2] sample/10BT/001_00000.parquet: already done, skipping

Report: /content/drive/MyDrive/chotu/test/reports/data_report.md
Train tokens: 0.00B


In [ ]:
# 4. Full run (a few hours, resumable)
!cd /content/Chotu && python scripts/prepare_data.py --storage /content/drive/MyDrive/chotu

14 source files, storage=/content/drive/MyDrive/chotu, workers=2
[OK] round trip: 200/200 documents identical after encode-decode
[OK] digits split: 2048 -> ['2', '0', '4', '8']
[OK] <|im_start|> single token: ids=[1]
[OK] <|im_end|> single token: ids=[2]
[OK] <|endoftext|> single token: ids=[0]
[OK] end-of-text id is 0: 
[OK] vocab fits uint16: vocab=49,152
[1/14] sample/10BT/000_00000.parquet: already done, skipping

sample/10BT/001_00000.parquet: downloading bytes:  12% 267M/2.15G [00:01<00:04, 407MB/s, 19.0MB/s  ]
sample/10BT/001_00000.parquet: downloading bytes:  24% 521M/2.15G [00:01<00:03, 502MB/s, 42.0MB/s  ]
sample/10BT/001_00000.parquet: downloading bytes:  29% 619M/2.15G [00:01<00:03, 493MB/s, 53.3MB/s  ]
sample/10BT/001_00000.parquet: downloading bytes:  59% 1.27G/2.15G [00:03<00:02, 434MB/s,  107MB/s  ]
sample/10BT/001_00000.parquet: downloading bytes:  77% 1.66G/2.15G [00:06<00:02, 233MB/s,  105MB/s  ]
sample/10BT/001_00000.parquet: downloading bytes:  83% 1.78G/2.15G [00

In [ ]:
# 5. Read the report: token count, tokenizer checks, 10 samples
from IPython.display import Markdown
Markdown(open('/content/drive/MyDrive/chotu/reports/data_report.md').read())

## Gate
- About 10B train tokens, and a validation split exists
- You read the 10 samples: clean English, documents separated, no markup junk

Then add your entry to `docs/PROGRESS.md` (numbers from the report) and push.

In [ ]:
!cd /content/Chotu && git add docs/ && git commit -m "Session 2: data prepared" && git pull --rebase -q && git push